In [2]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
import fastf1
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType
from functools import reduce

spark = SparkSession.builder \
        .appName("F1_Telemetry_Processing") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
        .getOrCreate()

url = "jdbc:postgresql://localhost:5432/telemetry"
properties = {"user": "michal", "password": "root", "driver": "org.postgresql.Driver"}

In [6]:
# --- dim_season ---
# CREATE TABLE dim_season(
#     season_id serial primary key,
#     year int not null
# );
dim_season = spark.createDataFrame([
        {"year": 2019},
        {"year": 2020},
        {"year": 2021},
        {"year": 2022},
        {"year": 2023},
        {"year": 2024},
        {"year": 2025},
    ])
dim_season.write.jdbc(url=url, table='dim_season', mode='append', properties=properties)

In [7]:
# --- dim_circuit ---
# CREATE TABLE dim_circuit(
#     circuit_id serial primary key,
#     name varchar(80) not null,
#     country varchar(80) not null,
#     location varchar(80) not null,
#     length_km real not null,
#     corners int not null

# );
dim_circuit = spark.createDataFrame([
        {"name": 'Albert Park Circuit', 'country': 'Australia', 'location': 'Melbourne', 'length_km': 5.278, 'corners': 16},
        {"name": 'Bahrain International Circuit', 'country': 'Bahrain', 'location': 'Sakhir', 'length_km': 5.412, 'corners': 15},
        {"name": 'Shanghai International Circuit', 'country': 'China', 'location': 'Shanghai', 'length_km': 5.451, 'corners': 16},
        {"name": 'Baku City Circuit', 'country': 'Azerbaijan', 'location': 'Baku', 'length_km': 6.003, 'corners': 20},
        {"name": 'Circuit de Barcelona-Catalunya', 'country': 'Spain', 'location': 'Barcelona', 'length_km': 4.657, 'corners': 14},
        {"name": 'Circuit de Monaco', 'country': 'Monaco', 'location': 'Monte Carlo', 'length_km': 3.337, 'corners': 19},
        {"name": 'Circuit Gilles-Villeneuve', 'country': 'Canada', 'location': 'Montréal', 'length_km': 4.361, 'corners': 14},
        {"name": 'Circuit Paul Ricard', 'country': 'France', 'location': 'Le Castellet', 'length_km': 5.842, 'corners': 15},
        {"name": 'Red Bull Ring', 'country': 'Austria', 'location': 'Spielberg', 'length_km': 4.318, 'corners': 10},
        {"name": 'Silverstone Circuit', 'country': 'Great Britain', 'location': 'Silverstone', 'length_km': 5.891, 'corners': 18},
        {"name": 'Hockenheimring', 'country': 'Germany', 'location': 'Hockenheim', 'length_km': 4.574, 'corners': 16},
        {"name": 'Hungaroring', 'country': 'Hungary', 'location': 'Budapest', 'length_km': 4.381, 'corners': 14},
        {"name": 'Circuit de Spa-Francorchamps', 'country': 'Belgium', 'location': 'Spa-Francorchamps', 'length_km': 7.004, 'corners': 20},
        {"name": 'Autodromo Nazionale di Monza', 'country': 'Italy', 'location': 'Monza', 'length_km': 5.793, 'corners': 11},
        {"name": 'Marina Bay Street Circuit', 'country': 'Singapore', 'location': 'Singapore', 'length_km': 4.94, 'corners': 19},
        {"name": 'Sochi Autodrom', 'country': 'Russia', 'location': 'Sochi', 'length_km': 5.848, 'corners': 15},
        {"name": 'Suzuka International Racing Course', 'country': 'Japan', 'location': 'Suzuka', 'length_km': 5.807, 'corners': 18},
        {"name": 'Autódromo Hermanos Rodríguez', 'country': 'Mexico', 'location': 'Mexico City', 'length_km': 4.304, 'corners': 17},
        {"name": 'Circuit of the Americas', 'country': 'United States', 'location': 'Austin', 'length_km': 5.513, 'corners': 20},
        {"name": 'Autodromo José Carlos Pace', 'country': 'Brazil', 'location': 'São Paulo', 'length_km': 4.309, 'corners': 15},
        {"name": 'Yas Marina Circuit', 'country': 'Abu Dhabi', 'location': 'Yas Island', 'length_km': 5.281, 'corners': 16},
        {"name": 'Autodromo Internazionale del Mugello', 'country': 'Italy', 'location': 'Mugello', 'length_km': 5.245, 'corners': 15},
        {"name": 'Nürburgring', 'country': 'Germany', 'location': 'Nürburg', 'length_km': 5.148, 'corners': 15},
        {"name": 'Algarve International Circuit', 'country': 'Portugal', 'location': 'Portimão', 'length_km': 4.653, 'corners': 15},
        {"name": 'Autodromo Internazionale Enzo e Dino Ferrari', 'country': 'Italy', 'location': 'Imola', 'length_km': 4.909, 'corners': 19},
        {"name": 'Intercity Istanbul Park', 'country': 'Turkey', 'location': 'Istanbul', 'length_km': 5.338, 'corners': 14},
        {"name": 'Circuit Zandvoort', 'country': 'Netherlands', 'location': 'Zandvoort', 'length_km': 4.259, 'corners': 14},
        {"name": 'Lusail International Circuit', 'country': 'Qatar', 'location': 'Lusail', 'length_km': 5.419, 'corners': 16},
        {"name": 'Jeddah Corniche Circuit', 'country': 'Saudi Arabia', 'location': 'Jeddah', 'length_km': 6.174, 'corners': 27},
        {"name": 'Miami International Autodrome', 'country': 'United States', 'location': 'Miami', 'length_km': 5.412, 'corners': 19},
        {"name": 'Las Vegas Strip Circuite', 'country': 'United States', 'location': 'Las Vegas', 'length_km': 6.201, 'corners': 17}
    ])
dim_circuit.write.jdbc(url=url, table='dim_circuit', mode='append', properties=properties)

In [48]:
# --- dim_event ---
# CREATE TABLE dim_event(
#     event_id serial primary key,
#     season_id int not null,
#     circuit_id int not null,
#     round_number int not null,
#     gp_name varchar(100),
#     event_date date,
#     event_format varchar(20),
#     CONSTRAINT fk_season_event FOREIGN KEY (season_id)
#     REFERENCES dim_season(season_id)
#     ON DELETE RESTRICT
#     ON UPDATE CASCADE,
#     CONSTRAINT fk_circuit_event FOREIGN KEY (circuit_id)
#     REFERENCES dim_circuit(circuit_id)
#     ON DELETE RESTRICT
#     ON UPDATE CASCADE
# );

dim_season = spark.read.jdbc(url=url, table='dim_season', properties=properties)
season_map = {row['year']: row['season_id'] for row in dim_season.collect()}

dim_circuit = spark.read.jdbc(url=url, table='dim_circuit', properties=properties)
circuit_map_location = {row["location"]: row["circuit_id"] for row in dim_circuit.collect()}
circuit_map_country  = {row["country"]:  row["circuit_id"] for row in dim_circuit.collect()}

mapping_seasons = F.create_map([F.lit(k) for pair in season_map.items() for k in pair])

@F.udf(IntegerType())
def map_circuit(location, country):
    return (
        circuit_map_location.get(location) or
        circuit_map_country.get(country)
    )

all_events = []
for year in range(2019, 2026):
    event = fastf1.get_event_schedule(year)
    event = spark.createDataFrame(pd.DataFrame(event))
    event = event.withColumn('season_id', mapping_seasons[year])
    event = event.withColumn('circuit_id', map_circuit(F.col('location'), F.col('Country')))
    event = event.withColumn("EventDate", F.date_format(F.col("EventDate"), "yyyy-MM-dd").cast(DateType()))
    event = event.select(['season_id', 'circuit_id', 'RoundNumber', 'OfficialEventName', 'EventDate', 'EventFormat'])
    event = event.withColumnsRenamed({'RoundNumber': 'round_number', 'OfficialEventName': 'gp_name', 'EventDate': 'event_date', 'EventFormat': 'event_format'})
    all_events.append(event)
    
dim_event = reduce(lambda a, b: a.union(b), all_events)
dim_event.write.jdbc(url=url, table='dim_event', mode='append', properties=properties)    

In [70]:
# --- dim_session ---
# CREATE TABLE dim_session(
#     session_id serial primary key,
#     event_id int not null,
#     session_type varchar(3),
#     session_date date,
#     CONSTRAINT fk_session_event FOREIGN KEY (event_id)
#     REFERENCES dim_event(event_id)
#     ON DELETE RESTRICT
#     ON UPDATE CASCADE
# );
dim_season = spark.read.jdbc(url=url, table='dim_season', properties=properties)
season_map = {row['year']: row['season_id'] for row in dim_season.collect()}

dim_event = spark.read.jdbc(url=url, table='dim_event', properties=properties)   
all_sessions = []

event_map = {
    (row["season_id"], row["round_number"]): (row["event_id"], row["event_format"])
    for row in dim_event.collect()
}

session_map = {
    'Practice 1': 'FP1',
    'Practice 2': 'FP2',
    'Practice 3': 'FP3',
    'Qualifying': 'Q',
    'Race': 'R',
    'Sprint Qualifying': 'SQ',
    'Sprint': 'S',
    'Sprint Shootout': 'SS',
    '': ''
}

for year in range(2019, 2026):
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    for _, event_row in schedule.iterrows():
        season_id = season_map[event_row["EventDate"].year]
        round_number = int(event_row["RoundNumber"])
    
        event_id, event_format = event_map.get((season_id, round_number))
        for session_num in range(1,6):
            ts = event_row[f'Session{session_num}Date']
            all_sessions.append({
                "event_id":      event_id,
                "session_type":  session_map[event_row[f'Session{session_num}']],
                "session_date":  ts.strftime("%Y-%m-%d") if pd.notna(ts) else None
            })
dim_session = spark.createDataFrame(pd.DataFrame(all_sessions))
dim_session = dim_session.withColumn("session_date", F.date_format(F.col("session_date"), "yyyy-MM-dd").cast(DateType()))

dim_session.write.jdbc(url=url, table='dim_session', mode='append', properties=properties)    